# Lab 5.11 &mdash; Reflection: Self-Critique and Revise

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Produce a (deliberately wrong) first answer
- Run a critic that checks the answer against a rule
- Revise when the critic objects, and confirm the fix

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; Planning patterns at a glance](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-11"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
**Reflection** has the agent **critique its own output** and improve it before answering. We seed a
**wrong first attempt** (it forgets to double the population), run a **critic** that catches the
mistake, and **revise**. Reflection trades extra work for higher quality &mdash; here it turns a
wrong answer into a right one.

In [ ]:
# DEMO -- task: report TWICE the population of Metropolis. The CORRECT answer is DERIVED
# from the looked-up base (not memorised) -- so a good critic reasons about a rule, it does not
# just match one constant. Change the base below and a rule-based critic still works; a hardcoded
# "== 240000" would silently break.
KB = {"population of metropolis": 120000}
def base_population():
    return KB["population of metropolis"]
def first_attempt(goal):
    # a naive agent that returns the base but forgets to double -- deliberately wrong
    return base_population()
print("naive first answer:", first_attempt("twice the population"), "(the base, not doubled)")

## Your Turn
Implement the critic, the revision, and the reflect loop. The critic must check a **derived rule**
&mdash; *the answer equals twice the looked-up base* &mdash; **not** a hardcoded `240000`. Derive the
expected value so the critic would still be right if the base population changed.

In [ ]:
KB = {"population of metropolis": 120000}
def base_population():
    return KB["population of metropolis"]
def first_attempt(goal):
    return base_population()   # deliberately wrong (forgot to double)

def critic(answer):
    # return (ok, feedback). Derive the target from the base -- do not hardcode 240000.
    want = ___    # TODO: twice the looked-up base population (a derived value)
    if answer == want:
        return (True, "looks correct")
    return (False, "expected twice the base population")

def revise(answer, feedback):
    return ___    # TODO: double the answer to fix the mistake

def reflective_answer(goal):
    ans = first_attempt(goal)
    ok, feedback = critic(ans)
    if ___:    # TODO: revise only when the critic REJECTED the answer
        ans = revise(ans, feedback)
    return {"final": ans, "critic_fired": not ok}

try:
    out = reflective_answer('twice the population of metropolis')
    print('final:', out['final'], '| critic fired:', out['critic_fired'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("reflection corrects the final answer", lambda: reflective_answer("x")["final"] == 2 * base_population())
expect_true("the critic fired on the wrong first answer", lambda: reflective_answer("x")["critic_fired"] is True)
expect_true("critic accepts the derived-correct answer", lambda: critic(2 * base_population())[0] is True)
expect_true("critic rejects the un-doubled base", lambda: critic(base_population())[0] is False)
expect_true("the critic is DERIVED, not hardcoded (still right if the base changes)", lambda: (KB.__setitem__("population of metropolis", 500000), critic(1000000)[0] is True and critic(240000)[0] is False, KB.__setitem__("population of metropolis", 120000))[1])
expect_true("revise applies the fix", lambda: revise(120000, "double it") == 240000)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; swap the rule-based policy for a REAL LLM (not graded)
Let a REAL LLM be the critic; the answer -> critique -> revise loop above is identical. Safe to skip &mdash; it needs a local **Ollama** (`pip install langchain-ollama`, then
`ollama run llama3.2:1b`) or a **Groq** key (`pip install langchain-groq`, `GROQ_API_KEY`). If
neither is present the cell prints a friendly note and moves on. **The graded steps above run on a
deterministic rule-based policy, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    draft = "The answer is 120000."
    review = llm.invoke("The task was 'twice the population (120000) of Metropolis'. "
                        "Critique this answer in one line: " + draft).content
    print("LLM critic says:", review)
    print("A real LLM critic catches subtler mistakes; the reflect loop is unchanged.")
except Exception as e:
    print("No local LLM available -- skipping (install langchain-ollama + `ollama run llama3.2:1b`,", type(e).__name__)
    print("or use Groq: `from langchain_groq import ChatGroq` with GROQ_API_KEY).")
    print("The rule-based critic above already caught the wrong first answer and triggered a revision.")

---
### Key takeaway
Reflection = answer, critique, revise. It costs extra calls but lifts quality &mdash; reach for it when being right matters more than being fast.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>